In [1]:
import sys
sys.path.append('../')  # subir 1 nivel desde notebooks a src
import json
from src.core.scraper_agent import ScraperAgent
from src.core.validator_agent import ValidatorAgent
from src.core.writer_agent import WriterAgent
from src.sources.inventory.app import Inventory
from src.config.settings import COUNTRY

db_loader = Inventory()


In [2]:
df_data_base = db_loader.load_db_from_country_selected()

CO


In [3]:
df_data_base_test = df_data_base[df_data_base["code"] == "CO928-akt-motos-nkd-125"]
df_data_base_test

,date,code,brand,model,year,type,technical_specs,publication_url,publication_image_url
276,17/02/2026,CO928-akt-motos-nkd-125,AKT Motos,NKD 125,2025,Urbana,"[{'key': 'displacement', 'value': 124, 'type':...",https://www.galgo.com/co/motos/CO928-akt-motos...,https://images.ctfassets.net/8zlbnewncp6f/01rS...


In [4]:
for index, row in df_data_base_test.iterrows():
    BRAND = row["brand"]
    MODEL = row["model"]
    YEAR = "2025" # Hardcodeado para traer información de año pasado y no mas reciente ya que no hay
    TYPE = row["type"]
    COUNTRY = {"CO": "Colombia", "MX": "Mexico", "CL": "Chile"}.get(COUNTRY)
    TECHNICAL_SPECS = row["technical_specs"]
    print(f"Marca: {BRAND}, Modelo: {MODEL}, Año: {YEAR}, País: {COUNTRY}")

Marca: AKT Motos, Modelo: NKD 125, Año: 2025, País: Colombia


## 1. Scrapear data

In [5]:
scraper = ScraperAgent(brand=BRAND, model=MODEL, year=YEAR, country=COUNTRY)

In [6]:
# Consultar prompt armado
print(scraper.get_prompt_template())

# Agente Scraper - Recolección de Experiencias de Usuarios

## Objetivo

Recolectar experiencias reales y vivenciales de usuarios sobre la motocicleta AKT Motos NKD 125 2025 específicamente del mercado de Colombia.

**ENFOQUE PRINCIPAL**: Tu trabajo es capturar cómo se **siente** vivir con la moto y cómo la **perciben** los usuarios. Prioriza sensaciones, emociones, decisiones de compra y experiencias vividas, NO descripciones técnicas.

**IMPORTANTE**: Este agente SOLO recolecta información. NO filtra, NO valida contra ficha técnica, NO descarta. Solo recolecta, etiqueta correctamente y documenta con suficiente contexto para que las siguientes fases puedan trabajar.

---

## Qué Priorizar en la Recolección

### 1. Experiencias Vivenciales (Prioridad Máxima)

Captura información sobre:
- **Sensaciones de manejo**: estabilidad percibida, vibraciones, confianza al frenar, respuesta del acelerador, postura y comodidad
- **Problemas o dolores recurrentes**: fallas, defectos, inconvenientes

### Ejecutar búsqueda de información

In [7]:
experiencias_extraidas = scraper.scrape()

c:\Users\JTRUJILLO\Documents\Galgo\Scripts\Otros\deep_research_models\notebooks\..\src\core\gemini_processor.py:9: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = self.client.interactions.create(


In [8]:
experiencias_extraidas

{'experiencias_usuarios': [{'fuente': 'https://www.youtube.com/watch?v=CarlosEnMoto_Consumo',
   'pais_identificado': 'Colombia',
   'tipo_contenido': 'video_youtube',
   'fecha_aprox': '2025-10',
   'extractos_relevantes': [{'categoria': 'consumo_real',
     'texto': 'Rendimiento promedio de 123,39 kms/galón en prueba mixta (ciudad y subida al Alto de Minas).',
     'frecuencia_indicador': 'unica'},
    {'categoria': 'experiencia_propiedad',
     'texto': 'Esperaba un poco más de rendimiento, aunque sigue siendo un consumo bajo ideal para quienes necesitan una moto barata de mantener.',
     'frecuencia_indicador': 'media'},
    {'categoria': 'sensaciones_manejo',
     'texto': 'Probada subiendo el Alto de Minas, ritmo normal.',
     'frecuencia_indicador': 'baja'}],
   'menciones_specs_tecnicas': [{'spec': 'consumo',
     'mencion': '123,39 kms/galón',
     'contexto_mencion': 'confirma_tiene',
     'fuente_mencion': 'experiencia_propia'}],
   'relaciones_causa_efecto': [{'causa': 'P

## 2. Validar data extraída

In [9]:
validator = ValidatorAgent(brand=BRAND, model=MODEL, year=YEAR, experiencias=experiencias_extraidas.get("experiencias_usuarios"), ficha_tecnica=TECHNICAL_SPECS, country=COUNTRY)

In [10]:
# Consultar prompt armado
print(validator.get_prompt_template())

# Agente Validator - Validación de Experiencias (Nivel 1)

## Objetivo

Validar que las experiencias de usuarios correspondan al modelo AKT Motos NKD 125 2025 del país Colombia, comparándolas con la ficha técnica oficial.

**IMPORTANTE:** Esta es la validación automática (Nivel 1). Los casos con contradicciones se marcarán para re-research (Nivel 2).

---

## Ficha Técnica de Referencia

Marca: AKT Motos
Modelo: NKD 125
Año: 2025
País: Colombia

Especificaciones Técnicas:
CAMPOS CRÍTICOS:
- Freno delantero: Disco
- Freno trasero: Tambor

IMPORTANTES PARA EXPERIENCIA DE USUARIO:
- Cilindrada: 124 cc
- Capacidad del tanque: 9.8 litros
- Peso total: 94 kg

ÚTILES COMO CONTEXTO:
- Tipo de transmisión: Mecánica
- Número de cambios: 5 velocidades
- Suspensión trasera: Doble Amortiguador Lateral
- Tipo de motor: 4 tiempos
- Suspensión delantera: Horquilla Telescópica

---

## Experiencias a Validar

[
  {
    "fuente": "https://www.youtube.com/watch?v=CarlosEnMoto_Consumo",
    "pais_identifi

### Validar información extraída

In [11]:
# Validar lo que haya que hacerle nueva búsqueda
resultado_validacion = validator.validate()

# Obtener experiencias que requieren re-research
experiencias_re_research = resultado_validacion.get("experiencias_requieren_re_research", [])

In [12]:
# Lista para almacenar resultados del re-research
experiencias_verificadas = []
experiencias_excluidas = []

for exp_re_research in experiencias_re_research:
    flag = exp_re_research.get("flag")
    experiencia_completa = exp_re_research.get("experiencia_completa", {})

    print(f"\nRe-research para: {experiencia_completa.get('fuente', 'N/A')}")
    print(f"Flag: {flag}")

    # Ejecutar re-research
    resultado_re = validator.re_research(
        experiencia=experiencia_completa,
        ficha_tecnica=TECHNICAL_SPECS,
        marca=BRAND,
        modelo=MODEL,
        año=YEAR,
        pais=COUNTRY,
        flag=flag
    )

    # Actualizar experiencia con resultado del re-research
    experiencia_actualizada = validator.update_experience_after_re_research(
        experiencia_original=experiencia_completa,
        resultado_re_research=resultado_re
    )

    # Clasificar según resultado
    resultado = resultado_re.get("resultado", "")
    if resultado in ["INCLUIR", "INCLUIR_CON_NOTA"]:
        experiencias_verificadas.append(experiencia_actualizada)
    elif resultado in ["EXCLUIR", "EXCLUIR_POR_PRECAUCION"]:
        experiencias_excluidas.append({
            "experiencia": experiencia_actualizada,
            "razon": resultado_re.get("razon", "")
        })

    # print(f"Resultado: {resultado}")
    # print(f"Razón: {resultado_re.get('razon', 'N/A')}")

In [13]:
experiencias_finales = {
    "experiencias_validadas": resultado_validacion.get("experiencias_validadas", []),
    "experiencias_verificadas_incluir": experiencias_verificadas,
    "experiencias_excluidas": (
        resultado_validacion.get("experiencias_excluidas_automatico", []) +
        experiencias_excluidas
    )
}

In [14]:
experiencias_finales

{'experiencias_validadas': [{'fuente': 'https://www.youtube.com/watch?v=CarlosEnMoto_Consumo',
   'pais_confirmado': True,
   'version_correcta': True,
   'confidence': 95,
   'extractos': [{'categoria': 'consumo_real',
     'texto': 'Rendimiento promedio de 123,39 kms/galón en prueba mixta (ciudad y subida al Alto de Minas).',
     'frecuencia_indicador': 'unica'},
    {'categoria': 'experiencia_propiedad',
     'texto': 'Esperaba un poco más de rendimiento, aunque sigue siendo un consumo bajo ideal para quienes necesitan una moto barata de mantener.',
     'frecuencia_indicador': 'media'},
    {'categoria': 'sensaciones_manejo',
     'texto': 'Probada subiendo el Alto de Minas, ritmo normal.',
     'frecuencia_indicador': 'baja'}],
   'menciones_specs_tecnicas': [{'spec': 'consumo',
     'mencion': '123,39 kms/galón',
     'contexto_mencion': 'confirma_tiene',
     'fuente_mencion': 'experiencia_propia'}]},
  {'fuente': 'https://www.youtube.com/watch?v=AChancletear_Review',
   'pais_

### Exportar resultados

In [15]:
if COUNTRY == "Colombia":
    pais = "CO"
elif COUNTRY == "Mexico":
    pais = "MX"
elif COUNTRY == "Chile":
    pais = "CL"

In [16]:
import json
nombre_archivo = f'../src/data/models_data_recollected/{pais}-{BRAND}_{MODEL}.json'
with open(nombre_archivo, 'w', encoding='utf-8') as f:
    json.dump(experiencias_finales, f, ensure_ascii=False, indent=2)

print(f"\nResultados guardados en '{nombre_archivo}'")


Resultados guardados en '../src/data/models_data_recollected/CO-AKT Motos_NKD 125.json'


## 3. Escritor

In [17]:
if COUNTRY == "Colombia":
    pais = "CO"
elif COUNTRY == "Mexico":
    pais = "MX"
elif COUNTRY == "Chile":
    pais = "CL"

### Leer información recolectada

In [18]:
nombre_archivo = f'../src/data/models_data_recollected/{pais}-{BRAND}_{MODEL}.json'
with open(nombre_archivo, 'r', encoding='utf-8') as f:
    experiencias_finales = json.load(f)

In [19]:
# Obtener experiencias validadas y verificadas
experiencias_validadas = experiencias_finales.get("experiencias_validadas", [])
experiencias_verificadas = experiencias_finales.get("experiencias_verificadas_incluir", [])

### Generar base de conocimiento

In [20]:
# Preparar ficha técnica (mismo formato que para validator)
ficha_tecnica_dict = {
    "brand": BRAND,
    "model": MODEL,
    "year": str(YEAR),
    "country": COUNTRY,
    "technical_specs": TECHNICAL_SPECS  # String raw
}

writer = WriterAgent(brand=BRAND, model=MODEL, year=YEAR, tipo=TYPE, ficha_tecnica=TECHNICAL_SPECS, experiencias_validadas=experiencias_validadas, experiencias_verificadas=experiencias_verificadas, country=COUNTRY)

# Generar KB
knowledge_base = writer.write()

In [21]:
print(knowledge_base)

[SEGMENTO]

Tipo según BD: Urbana
Segmento identificado por usuarios: Base para personalización y transporte económico (Coincide con tipo declarado).

[SENTIMIENTO]

La NKD 125 2025 es percibida en Colombia como una "moto lienzo": una plataforma mecánica sencilla, extremadamente económica de mantener y fácil de modificar. Genera una conexión emocional por su estética clásica y respuesta alegre (torque), pero los usuarios aceptan explícitamente sacrificar comodidad y refinamiento (vibraciones y suspensión dura) a cambio de su bajo precio y versatilidad.

[SENSACIONES]

Estabilidad: Se percibe como una moto muy liviana (94 kg en ficha), lo que facilita la maniobrabilidad en el tráfico, aunque su torque tiende a levantar la rueda delantera (wheelie) si se acelera bruscamente.

Vibraciones: Altas y constantes. Los usuarios describen la experiencia como tener "masajeador incluido" debido a la transmisión de vibraciones del motor al chasis y manubrio, siendo notablemente más vibrante que com

In [22]:
nombre_archivo = f'../src/data/output/gemini/{pais}-{BRAND}_{MODEL}-knowledge_base.md'
with open(nombre_archivo, 'w', encoding='utf-8') as f:
    f.write(knowledge_base)

## Validar informe